# 26 · India school access: nearest school and E2SFCA per national level — Part D, step D5 (schools, part 3)

**Same method as notebook 25 (Kerala)**, for all of India: 140 M populated 100 m cells (WorldPop 2026) × 1.07 M
Government + Aided schools (UDISE 2022, corrected points), national levels primary (classes 1–5, 1 km), upper
primary (6–8, 3 km), secondary (9–10, 5 km), higher secondary (11–12, 7 km); Gaussian decay with σ = catchment
(Dai 2010); main run = norm-based places (RTE / RMSA × teachers), sensitivity = pupils enrolled.

**State borders.** Children use schools in a neighbouring state and the other way round, so E2SFCA runs in two
passes, each state by state:
1. **Pass 1, per state of the school:** each school's ratio R_j = places ÷ distance-weighted children, counting the
   cells of every state within its catchment (the state's school bounding box + 0.09°, ≥ 7 km).
2. **Pass 2, per state of the cell:** each cell's score A_i = Σ R_j · w_ij over the schools of any state in reach;
   nearest school of the level anywhere in India (KD-tree).

**Restartable:** pass 1 writes one file per state, pass 2 one folder per state (with `_SUCCESS`); a re-run skips
what is done. Uttar Pradesh (13.75 M cells) runs first.

**Checks:** per state, KD-tree = brute force (200 cells) and catchment consistency; at the end, the national
**Σ score × children = Σ places reached** (exact only if no cell or school was missed at a border).

**Gaps stated:** UDISE 2022 has no schools for Dadra & Nagar Haveli and Daman & Diu (national share used; schools in
Gujarat / Maharashtra within reach still count); 0.093 M people in cells outside every state polygon are left out.

**Outputs** (`data/lake/gold/`): `school_access_india/state=…` (one row per cell), `school_ratio_india/` (per
school), `school_access_india_by_state.csv`, `…_by_district.csv`, `…_checks.csv`, `…_main_vs_enrol.csv`,
`…_progress.csv` (written during the run).

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Install the Part D packages

In [3]:
# Part D packages, pinned in requirements-bigdata.txt (PySpark 3.5.9, apache-sedona 1.9.1, pyarrow).
# Colab also needs Java 17, which Spark runs on (the laptop already has it: JAVA_HOME).
import os
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-bigdata.txt")], check=True)
if IN_COLAB:
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Start Spark + Sedona

In [4]:
import shutil
import time

import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
from bigdata.spark_session import get_spark

sys.path.insert(0, str(PROJECT / "scripts" / "bigdata"))
import population_cells as pc
import school_capacity as sc
from build_india_health_facilities import fold, haversine_m

SILVER, GOLD = PROJECT / "data" / "lake" / "silver", PROJECT / "data" / "lake" / "gold"
RATIO, ACCESS = GOLD / "school_ratio_india", GOLD / "school_access_india"
LEVELS = list(sc.INDIA_LEVELS)
VARS = ["main", "enrol"]
MARGIN = 0.09        # degrees around a bounding box: >= 7 km (largest catchment) in latitude, and in longitude up to 37 N
EARTH_R = 6371000.0  # Earth radius (m) of the project's haversine_m()
FIRST = "Uttar Pradesh"  # the largest state runs first: its checks are reviewed before the others matter
assert shutil.disk_usage(PROJECT).free / 1e9 > 8, "less than 8 GB free: stop"
spark = get_spark(PROJECT, cores=6, memory="6g")  # 6 cores: the laptop's free RAM (see notebook 17)
LON, LAT = pc.lonlat_sql(PROJECT / "data" / "raw" / "worldpop")
print("cores:", spark.sparkContext.defaultParallelism, "| Spark UI:", spark.sparkContext.uiWebUrl)

cores: 6 | Spark UI: http://127.0.0.1:4040


## Demand (cells) and supply (schools)

In [5]:
# Demand: children of each national level's ages x Government + Aided share of the cell's state (UDISE 2022)
share = pd.read_parquet(SILVER / "school_demand_share").query("scope == 'india'").assign(key=lambda d: d["state_name"].map(fold))
national = share.groupby("level")[["pupils_ga", "pupils_all"]].sum().eval("pupils_ga / pupils_all")
cells_all = (spark.read.parquet(str(SILVER / "population_cells")).filter("state <> '(outside boundaries)'")
             .withColumn("lon", F.expr(LON).cast("double")).withColumn("lat", F.expr(LAT).cast("double")))
states = cells_all.groupBy("state").agg(F.count("*").alias("cells"), F.sum("pop").alias("pop"), F.min("lon").alias("lon0"),
                                        F.max("lon").alias("lon1"), F.min("lat").alias("lat0"), F.max("lat").alias("lat1")).toPandas()
states = states.assign(key=states["state"].map(fold)).merge(share.pivot(index="key", columns="level", values="share"),
                                                          left_on="key", right_index=True, how="left")
no_udise = states.loc[states[LEVELS].isna().any(axis=1), "state"].tolist()
for lv in LEVELS:
    states[lv] = states[lv].fillna(national[lv])
print("cell states:", len(states), "| without UDISE schools (national share used; schools across the border still count):", no_udise)
share_df = spark.createDataFrame(states[["state", *LEVELS]].rename(columns={lv: f"sh_{lv}" for lv in LEVELS}))
cells = (cells_all.join(F.broadcast(share_df), "state")
         .selectExpr("cell_id", "state", "district_code", "subdistrict_code", "pop", "lon", "lat", "ST_Point(lon, lat) AS geom",
                     *[f"coalesce({sc.demand_sql(lv)}, 0) * sh_{lv} AS dem_{lv}" for lv in LEVELS]))

# Supply: places per school and level (notebook 24); sid = a row id (16 Kerala schools share a UDISE code)
sup_pd = pd.read_parquet(SILVER / "school_supply_india").reset_index(drop=True)
assert not sup_pd.duplicated(["school_id", "level", "lat", "lon", "cap_main"]).any() or        sup_pd.duplicated(["school_id", "level"]).sum() < 100, "supply rows duplicated: re-run notebook 24"
sup_pd["state_name"] = sup_pd["state_name"].astype(str)
sup_pd["sid"] = np.arange(len(sup_pd))
sup = spark.createDataFrame(sup_pd[["sid", "level", "state_name", "lat", "lon", "cap_main", "cap_enrol"]]) \
           .selectExpr("*", "ST_Point(lon, lat) AS geom").cache()
print(f"{states['cells'].sum():,} cells, {states['pop'].sum() / 1e6:,.1f} M people | school-level rows:", sup_pd.groupby("level").size().to_dict())
school_states = [FIRST] + sup_pd.loc[sup_pd["state_name"] != FIRST, "state_name"].value_counts(ascending=True).index.tolist()
cell_states = [FIRST] + states.loc[states["state"] != FIRST].sort_values("cells")["state"].tolist()

cell states: 36 | without UDISE schools (national share used; schools across the border still count): ['Dādra and Nagar Haveli and Damān and Diu']


140,078,322 cells, 1,470.2 M people | school-level rows: {'higher_secondary': 65427, 'primary': 908985, 'secondary': 144301, 'upper_primary': 403895}


## Pairs, nearest school

In [6]:
def cells_near(lon0, lon1, lat0, lat1):
    """Cells within a bounding box + MARGIN; only the partitions (states) that overlap it are read."""
    m = MARGIN
    hit = states[(states["lon1"] >= lon0 - m) & (states["lon0"] <= lon1 + m) & (states["lat1"] >= lat0 - m) & (states["lat0"] <= lat1 + m)]
    return cells.filter(F.col("state").isin(hit["state"].tolist()) & F.col("lon").between(lon0 - m, lon1 + m)
                        & F.col("lat").between(lat0 - m, lat1 + m))


def pairs(c, s, level, cols):
    """(cell, school) pairs within the level's catchment: broadcast index join (D4), ellipsoid distance, Gaussian weight."""
    c.createOrReplaceTempView("c")
    s.createOrReplaceTempView("s")
    d0 = sc.CATCHMENT_M[level]
    p = spark.sql(f"""SELECT /*+ BROADCAST(s) */ c.cell_id, c.dem_{level} AS dem, s.sid, {", ".join("s." + x for x in cols)},
                             ST_DistanceSpheroid(c.geom, s.geom) AS d
                      FROM c JOIN s ON ST_DWithin(c.geom, s.geom, {d0}, true)""")
    assert "BroadcastIndexJoin" in p._jdf.queryExecution().executedPlan().toString(), "no broadcast index join"
    return p.withColumn("w", F.expr(sc.gaussian("d", repr(float(d0)))))


def xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])


TREES = {lv: cKDTree(xyz(g["lat"].to_numpy(), g["lon"].to_numpy())) for lv, g in sup_pd.groupby("level")}


def nearest(c, level):
    """Great-circle distance (m) from each cell to the nearest school of the level, all of India (KD-tree in the tasks)."""
    tree = TREES[level]

    def f(batches):
        for b in batches:
            chord, _ = tree.query(xyz(b["lat"].to_numpy(), b["lon"].to_numpy()))
            yield pd.DataFrame({"cell_id": b["cell_id"], f"nearest_m_{level}": 2 * EARTH_R * np.arcsin(np.minimum(chord / 2, 1))})
    return c.select("cell_id", "lat", "lon").mapInPandas(f, f"cell_id long, nearest_m_{level} double")

## Pass 1: school ratios, state by state

In [7]:
# Pass 1, per state of the school: R_j = places_j / sum_i(children_i x w_ij), with the cells of every state in reach
RATIO.mkdir(parents=True, exist_ok=True)
for st in school_states:
    f = RATIO / f"{st}.parquet"
    if f.exists():
        continue
    t, parts = time.time(), []
    for lv in LEVELS:
        g = sup_pd[(sup_pd["state_name"] == st) & (sup_pd["level"] == lv)]
        if g.empty:
            continue
        s = sup.filter((F.col("state_name") == st) & (F.col("level") == lv))
        p = pairs(cells_near(g["lon"].min(), g["lon"].max(), g["lat"].min(), g["lat"].max()), s, lv, ["cap_main"])
        parts.append(p.groupBy("sid").agg(F.sum(F.col("dem") * F.col("w")).alias("weighted_children"),
                                          F.count("*").alias("cells_in_catchment")).toPandas())
    r = pd.concat(parts).merge(sup_pd[["sid", "school_id", "level", "state_name", "cap_main", "cap_enrol"]], on="sid")
    for v in VARS:
        r[f"r_{v}"] = r[f"cap_{v}"] / r["weighted_children"].where(r["weighted_children"] > 0)
    r.to_parquet(f.with_suffix(".tmp"), index=False)
    f.with_suffix(".tmp").rename(f)  # a state counts as done only when its file is complete
    print(f"pass 1 {st:42s} {len(r):8,} school-levels with children in reach  {time.time() - t:6.0f} s", flush=True)
ratios = pd.concat([pd.read_parquet(RATIO / f"{st}.parquet") for st in school_states], ignore_index=True)
print("school-levels with children in reach:", len(ratios), "of", len(sup_pd))

pass 1 Uttar Pradesh                               188,025 school-levels with children in reach     321 s


pass 1 Lakshadweep                                      19 school-levels with children in reach      20 s


pass 1 Chandigarh                                      401 school-levels with children in reach      27 s


pass 1 Puducherry                                      503 school-levels with children in reach      68 s


pass 1 Andaman and Nicobar Islands                     592 school-levels with children in reach      21 s


pass 1 Sikkim                                        1,097 school-levels with children in reach      28 s


... (26 outputs left out)

pass 1 Madhya Pradesh                              118,204 school-levels with children in reach     237 s


pass 1 Rajasthan                                   129,824 school-levels with children in reach     339 s


pass 1 Maharashtra                                 143,202 school-levels with children in reach     281 s


school-levels with children in reach: 1520166 of 1522608


## Pass 2: cell scores and nearest school, state by state (+ checks)

In [8]:
# Pass 2, per state of the cell: A_i = sum_j(R_j x w_ij) over schools of any state in reach; nearest school; checks
sup_r = sup_pd[["sid", "level", "lat", "lon"]].merge(ratios[["sid", "r_main", "r_enrol"]], on="sid")
ACCESS.mkdir(parents=True, exist_ok=True)
log = []
for st in cell_states:
    out_dir = ACCESS / f"state={st}"
    if (out_dir / "_SUCCESS").exists():
        continue
    t, b = time.time(), states.set_index("state").loc[st]
    c = cells.filter(F.col("state") == st)
    out = c.select("cell_id", "district_code", "subdistrict_code", "pop", "lon", "lat", *[f"dem_{lv}" for lv in LEVELS]).cache()
    for lv in LEVELS:
        g = sup_r[(sup_r["level"] == lv) & sup_r["lon"].between(b["lon0"] - MARGIN, b["lon1"] + MARGIN)
                  & sup_r["lat"].between(b["lat0"] - MARGIN, b["lat1"] + MARGIN)]
        new = out
        if len(g):
            s = spark.createDataFrame(g[["sid", "lat", "lon", "r_main", "r_enrol"]]).selectExpr(
                "sid", "lat", "lon", *[f"nanvl(r_{v}, NULL) AS r_{v}" for v in VARS], "ST_Point(lon, lat) AS geom")  # NaN -> null
            agg = pairs(c, s, lv, ["r_main", "r_enrol"]).groupBy("cell_id").agg(
                *[F.sum(F.col(f"r_{v}") * F.col("w")).alias(f"a_{v}_{lv}") for v in VARS], F.count("*").alias(f"schools_in_reach_{lv}"))
            new = new.join(agg, "cell_id", "left")
        else:
            new = new.select("*", *[F.lit(None).cast("double").alias(f"a_{v}_{lv}") for v in VARS],
                             F.lit(None).cast("long").alias(f"schools_in_reach_{lv}"))
        new = new.join(nearest(c, lv), "cell_id", "left").cache()
        new.count()  # materialise this level once
        out.unpersist()
        out = new
    # checks: KD-tree = brute force (200 cells), and a cell in reach has its nearest school within the catchment
    # (ellipsoid catchment vs sphere distance: up to 0.54 % apart, allowed 0.6 %)
    smp = out.sample(fraction=min(1.0, 400 / b["cells"]), seed=1).limit(200).toPandas()
    for lv in LEVELS:
        s = sup_pd[sup_pd["level"] == lv]
        brute = np.array([haversine_m(la, lo, s["lat"].to_numpy(), s["lon"].to_numpy()).min() for la, lo in zip(smp["lat"], smp["lon"])])
        assert np.abs(smp[f"nearest_m_{lv}"] - brute).max() < 0.01, f"{st} {lv}: KD-tree differs from brute force"
        bad = out.filter(f"schools_in_reach_{lv} > 0 AND nearest_m_{lv} > 1.006 * {sc.CATCHMENT_M[lv]}").count()
        assert bad == 0, f"{st} {lv}: {bad} cells in reach with the nearest school beyond the catchment"
    out.write.mode("overwrite").option("compression", "zstd").parquet(str(out_dir))
    out.unpersist()
    log.append({"state": st, "cells": int(b["cells"]), "seconds": round(time.time() - t)})
    pd.DataFrame(log).to_csv(GOLD / "school_access_india_progress.csv", index=False)  # visible while the run goes on
    print(f"pass 2 {st:42s} {int(b['cells']):11,} cells  {time.time() - t:6.0f} s  (checks passed)", flush=True)

pass 2 Uttar Pradesh                               13,754,598 cells     703 s  (checks passed)


pass 2 Lakshadweep                                        779 cells     107 s  (checks passed)


pass 2 Chandīgarh                                      14,618 cells      88 s  (checks passed)


pass 2 Puducherry                                      43,274 cells      85 s  (checks passed)


pass 2 Dādra and Nagar Haveli and Damān and Diu        51,431 cells      82 s  (checks passed)


pass 2 Andaman and Nicobar Islands                     51,519 cells      83 s  (checks passed)


... (26 outputs left out)

pass 2 Gujarāt                                     10,071,419 cells     458 s  (checks passed)


pass 2 Madhya Pradesh                              11,528,051 cells     558 s  (checks passed)


pass 2 Mahārāshtra                                 13,878,319 cells     813 s  (checks passed)


pass 2 Rājasthān                                   16,890,420 cells     981 s  (checks passed)


## National check: places are conserved

In [9]:
# National check (the property E2SFCA guarantees): sum over all cells of score x children = places of the schools reached
acc = spark.read.parquet(str(ACCESS))
assert acc.select("state").distinct().count() == len(cell_states)
got = acc.agg(*[F.sum(F.col(f"a_{v}_{lv}") * F.col(f"dem_{lv}")).alias(f"{v}_{lv}") for v in VARS for lv in LEVELS]).first()
checks = pd.DataFrame([{"level": lv, "variant": v, "sum_score_x_children": got[f"{v}_{lv}"],
                        "places_reached": ratios.loc[(ratios["level"] == lv) & ratios[f"r_{v}"].notna(), f"cap_{v}"].sum()}
                       for lv in LEVELS for v in VARS])
checks["rel_diff"] = (checks["sum_score_x_children"] - checks["places_reached"]).abs() / checks["places_reached"]
print(checks.to_string(index=False))
assert (checks["rel_diff"] < 1e-6).all(), "places not conserved: a cell or school was missed at a state border"

           level variant  sum_score_x_children  places_reached     rel_diff
         primary    main          8.524672e+07    8.524672e+07 1.048802e-15
         primary   enrol          7.160897e+07    7.160897e+07 0.000000e+00
   upper_primary    main          5.088159e+07    5.088159e+07 1.317868e-15
   upper_primary   enrol          4.122159e+07    4.122159e+07 2.530425e-15
       secondary    main          2.290349e+07    2.290349e+07 1.626516e-15
       secondary   enrol          2.363901e+07    2.363901e+07 6.303632e-16
higher_secondary    main          1.274479e+07    1.274479e+07 0.000000e+00
higher_secondary   enrol          1.479860e+07    1.479860e+07 3.524257e-15


## Results

In [10]:
# Results per state and level (children-weighted), India total, and main vs enrolment run across districts
def area_table(keys):
    aggs = []
    for lv in LEVELS:
        d = F.col(f"dem_{lv}")
        aggs += [F.sum(d).alias(f"children_{lv}"),
                 (100 * F.sum(F.when(F.col(f"schools_in_reach_{lv}").isNull(), d).otherwise(0)) / F.sum(d)).alias(f"beyond_norm_%_{lv}"),
                 (F.sum(d * F.col(f"nearest_m_{lv}")) / F.sum(d) / 1000).alias(f"mean_nearest_km_{lv}"),
                 *[(F.sum(d * F.coalesce(F.col(f"a_{v}_{lv}"), F.lit(0))) / F.sum(d)).alias(f"a_{v}_{lv}") for v in VARS]]
    return acc.groupBy(*keys).agg(F.sum("pop").alias("pop"), *aggs).toPandas()


by_state = area_table(["state"]).sort_values("state")
by_district = area_table(["state", "district_code"])
by_district.insert(2, "district", by_district["district_code"].map(pc.load_units(SILVER, "district_code").set_index("code")["unit_name"]))
total = area_table([]).iloc[0]
print("India:", {lv: {"children_M": round(total[f"children_{lv}"] / 1e6, 2), "beyond_norm_%": round(total[f"beyond_norm_%_{lv}"], 2),
                       "mean_nearest_km": round(total[f"mean_nearest_km_{lv}"], 2), "places_per_child_main": round(total[f"a_main_{lv}"], 3),
                       "places_per_child_enrol": round(total[f"a_enrol_{lv}"], 3)} for lv in LEVELS})
cols = ["state"] + [f"{m}_{lv}" for lv in LEVELS for m in ("beyond_norm_%", "a_main")]
print(by_state[cols].round(3).to_string(index=False))
agree = []
for lv in LEVELS:
    x = by_district[by_district[f"children_{lv}"] >= 100]
    w = [set(x.nsmallest(10, f"a_{v}_{lv}")["district_code"]) for v in VARS]
    agree.append({"level": lv, "districts": len(x), "spearman_main_vs_enrol": round(spearmanr(x[f"a_main_{lv}"], x[f"a_enrol_{lv}"]).statistic, 3),
                  "worst10_overlap": len(w[0] & w[1])})
agree = pd.DataFrame(agree)
print(agree.to_string(index=False))

India: {'primary': {'children_M': np.float64(77.44), 'beyond_norm_%': np.float64(15.98), 'mean_nearest_km': np.float64(0.64), 'places_per_child_main': np.float64(1.101), 'places_per_child_enrol': np.float64(0.925)}, 'upper_primary': {'children_M': np.float64(50.33), 'beyond_norm_%': np.float64(2.96), 'mean_nearest_km': np.float64(0.96), 'places_per_child_main': np.float64(1.011), 'places_per_child_enrol': np.float64(0.819)}, 'secondary': {'children_M': np.float64(32.29), 'beyond_norm_%': np.float64(3.64), 'mean_nearest_km': np.float64(1.74), 'places_per_child_main': np.float64(0.709), 'places_per_child_enrol': np.float64(0.732)}, 'higher_secondary': {'children_M': np.float64(31.89), 'beyond_norm_%': np.float64(9.69), 'mean_nearest_km': np.float64(3.35), 'places_per_child_main': np.float64(0.4), 'places_per_child_enrol': np.float64(0.464)}}
                                   state  beyond_norm_%_primary  a_main_primary  beyond_norm_%_upper_primary  a_main_upper_primary  beyond_norm_%_se

## Save

In [11]:
by_state.to_csv(GOLD / "school_access_india_by_state.csv", index=False)
by_district.to_csv(GOLD / "school_access_india_by_district.csv", index=False)
checks.to_csv(GOLD / "school_access_india_checks.csv", index=False)
agree.to_csv(GOLD / "school_access_india_main_vs_enrol.csv", index=False)
ratios.to_csv(GOLD / "school_ratio_india.csv", index=False)
print("written gold/school_access_india (one folder per state) + 5 tables")
spark.stop()

written gold/school_access_india (one folder per state) + 5 tables
